### Import packages
#### Create connection to Database

In [ ]:
from sqlalchemy import create_engine 
import pandas as pd
import numpy as np

# Connect to the database 
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()

In [ ]:
from sqlalchemy import create_engine, inspect 
from sqlalchemy import create_engine, inspect
import pandas as pd
import os
# Connect to the database
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()
db_inspector = inspect(db_engine)

tables = db_inspector.get_table_names()
print("Table found: ", tables)

output_folder = "./exports"
os.makedirs(output_folder, exist_ok=True)
for table_name, df in dataFrames.items():
    file_path = os.path.join(output_folder)
    df.to_csv(f"./exports/{table_name}.csv", index = False)
    print(f"Exported: {file_path}")

#### Read table from database

In [ ]:
# HINTS
# Load data into DataFrame
# user Pandas to read data from a table into a DataFrame

In [ ]:
# # Approach 1: Use Pandas.read_sql_table to read all columns from 'customers' table
table_name = 'customers'
df = pd.read_sql_table(table_name=table_name, con=db_conn)
df.tail(5)

In [ ]:
# # Approach 2: Use Pandas.read_sql_query to read these columns
# table_name = 'customers'
# columns = ['CustomerId', 'FirstName', 'LastName', 'Phone', 'Email', 'SupportRepId']
df = pd.read_sql_query(sql='select CustomerId, FirstName, LastName, Phone, Email, SupportRepId from customers', con=db_conn)
df.tail(5)

In [11]:
# Function to save tables to CSV 
def save_table_to_csv(table_name, conn, output_file):
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(output_file, index=False)

In [ ]:
# Code from instructor 

import yaml
import io
config_file = 'config.yml'
f = open(config_file, 'r')
config = yaml.safe_load(f)
config

def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv')
    print('Completed!\n')

def get_connection(db_type, host):
    if db_type == 'sqlite':
        db_connection_string = f'sqlite:///{host}.db'
        db_engine = create_engine(url=db_connection_string)
        return db_engine.connect()
    elif db_type == 'Oracle':
        db_connection_string = 'Oracle://{host}:1234'
        return db_engine.connect()
db_conn = get_connection(**config.get('source').get('database'))
extract_table(table_name='albums', con=db_conn, folder_path='destination/config_driven')

In [ ]:
# Approach 0: Hard code the table names
tables = ['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'invoices', 'media_types', 'playlist_track', 'playlists', 'tracks']

for table_name in tables:
    print(table_name)
    save_table_to_csv(table_name=table_name, conn=db_conn, output_file=table_name + '.csv')

#### Config-Driven Ingestion

In [ ]:
# # HINTS
# # Read configs stored in the 'config.yml' file

# # Read yaml file
# # Package: yaml (pip install pyyaml)
# # Function: load / safe_load
# # Print it after loading

In [ ]:
# Use loop function to read tables within config.source.table
# Export output into CSV
# Name Convention: '<date>__<table_name>.csv'
# Path: chinook/config_driven/
# note: use os.makedirs() if path is not exists


In [ ]:
# Approach 1: Config-driven ingestion

import os
import yaml

config_file = 'config.yml'
outdir = 'chinook/config_driven/'
os.makedirs(outdir, exist_ok=True)

with open(config_file, 'r') as cfg:
    data = yaml.safe_load(cfg)
    print("Content in config: ", data)
    tables = data['source'].get('table')
    print(tables)

for table_name in tables:
    print(table_name)
    # save_table_to_csv(table_name=table_name, conn=db_conn, output_file=outdir + table_name + '.csv')
    save_table_to_csv(table_name=table_name, conn=db_conn, output_file=f"{outdir}/20251127_{table_name}.csv")

### Metadata-Driven Ingestion

In [ ]:
# # HINTS
# # Read metadata from the database inlcuding tables / columns
# sqlite_metadata_table = 'sqlite_master'
# sqlite_metadata_condition = "type = 'table'"
# metadata_sql = f""" select 1"""
# print(metadata_sql)
# table_df = pd.read_sql_query(metadata_sql)
# print(table_df)
# FROM INSTRUCTOR: select name from sqlite_master where 1=1 and type = 'table and name not like 'sqlite_%'

In [ ]:
# loop for each table from the DataFrame
# read and extract table
# save to path: chinook/metadata_driven/
# note: use os.makedirs() if path is not exists

In [ ]:
# Approach 2: Metadata-driven ingestion

import os

outdir = 'chinook/metadata_driven/'
os.makedirs(outdir, exist_ok=True)

df_table_names = pd.read_sql_query(sql='SELECT name FROM sqlite_sequence;', con=db_conn)
print("Table names in DataFrame: ", df_table_names)

array_table_names = df_table_names['name'].tolist()
print("Table names in array: ", array_table_names)

for table_name in array_table_names:
    print(table_name)
    # save_table_to_csv(table_name=table_name, conn=db_conn, output_file=outdir + table_name + '.csv')
    save_table_to_csv(table_name=table_name, conn=db_conn, output_file=f"{outdir}/20251127_{table_name}.csv")

In [ ]:
metadata_sql = """select name from sqlite_master where 1=1 and type = 'table'   and name not like 'sqlite_%'"""
table_df = pd.read_sql_query(metadata_sql, con=db_conn)
names = list(table_df['name'])
import os 


# loop for each table from the DataFrame
# read and extract table
# save to path: chinook/metadata_driven/
# note: use os.makedirs() if path is not exists
def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv', index=False)
    print('Completed!\n')
for name in names:
    extract_table(table_name=name, con=db_conn, folder_path='destination/metadata')